###Section C

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

In [0]:
def id_validations(id):
    value = str(id).strip().upper()
    return None if value == "" else value

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.sessions (
    session_id STRING,
    charger_id STRING,
    member_id STRING,
    started_at TIMESTAMP,
    ended_at TIMESTAMP,
    session_date DATE,
    status STRING,
    time_band STRING,
    energy_kwh DECIMAL(10,3),
    duration_minutes INT,
    idle_minutes INT,
    amount_paid DECIMAL(10,2),
    updated_at TIMESTAMP
) USING DELTA;
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.session_faults (
    session_id STRING,
    fault_seq INT,
    fault_code STRING
) USING DELTA;
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_sessions (
    session_id STRING,
    reject_reason STRING,
    updated_at TIMESTAMP,
    raw_record STRING,
    quarantined_at TIMESTAMP
) USING DELTA;
""")

###C1

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DecimalType, IntegerType, DateType
from pyspark.sql.functions import udf

validate_id = udf(id_validations, StringType())

started_at_parsed = F.coalesce(
    F.try_to_timestamp(F.col('started_at'), F.lit('yyyy-MM-dd HH:mm:ss')),
    F.try_to_timestamp(F.col('started_at'), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
)

ended_at_parsed = F.coalesce(
    F.try_to_timestamp(F.col('ended_at'), F.lit('yyyy-MM-dd HH:mm:ss')),
    F.try_to_timestamp(F.col('ended_at'), F.lit("yyyy-MM-dd'T'HH:mm:ss"))
)

session_date_col = F.to_date(started_at_parsed)

bronze_raw = spark.read.table(f"{catalog}.bronze.sessions")

session_df = bronze_raw \
    .withColumn('raw_record', F.to_json(F.struct(*[F.col(c) for c in bronze_raw.columns]))) \
    .withColumn('meter_parsed', F.from_json(F.col('meter'), 'start_kwh double, end_kwh double')) \
    .withColumn('payment_parsed', F.from_json(F.col('payment'), 'method string, amount_inr double')) \
    .withColumn('faults_parsed', F.from_json(F.col('faults'), 'array<string>')) \
    .select(
        validate_id(F.col("member_id")).alias("member_id"),
        validate_id(F.col("charger_id")).alias("charger_id"),
        validate_id(F.col("session_id")).alias("session_id"),
        F.upper(F.trim(F.col("status"))).alias("status"),
        started_at_parsed.alias('started_at'),
        ended_at_parsed.alias('ended_at'),
        session_date_col.alias('session_date'),
        F.try_to_timestamp(F.col("updated_at")).alias('updated_at'),
        F.when(
            F.col('meter_parsed.end_kwh').isNotNull() & F.col('meter_parsed.start_kwh').isNotNull(),
            F.col('meter_parsed.end_kwh') - F.col('meter_parsed.start_kwh')
        ).otherwise(F.lit(None)).alias('energy_kwh'),
        F.when(
            started_at_parsed.isNotNull() & ended_at_parsed.isNotNull(),
            F.floor((ended_at_parsed.cast('long') - started_at_parsed.cast('long')) / 60)
        ).otherwise(F.lit(None)).alias('duration_minutes'),
        F.when(
            F.hour(started_at_parsed).between(18, 21),
            F.lit('PEAK')
        ).otherwise(F.lit('OFF_PEAK')).alias('time_band'),
        F.upper(F.trim(F.col('payment_parsed.method'))).alias('payment_method'),
        F.col('payment_parsed.amount_inr').cast('decimal(10,2)').alias('payment_amount_inr'),
        F.col('faults_parsed').alias('faults'),
        F.coalesce(F.col('idle_minutes').cast('int'), F.lit(0)).alias('idle_minutes'),
        F.col('raw_record')
    )

C2

In [0]:
from pyspark.sql.window import Window

window_fn = Window.partitionBy("session_id").orderBy(F.col("updated_at").desc())
de_dup_session_df = (
    session_df.withColumn("rank", F.row_number().over(window_fn))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

###C3

In [0]:
chargers_df = spark.read.table(f"{catalog}.silver.chargers")
members_df = spark.read.table(f"{catalog}.silver.members")

# Left join to verify reference table existence
validated_df = de_dup_session_df \
    .join(chargers_df.select(F.col('charger_id').alias('c_id'), F.lit(True).alias('charger_exists')),
          F.col('charger_id') == F.col('c_id'), 'left') \
    .join(members_df.select(F.col('member_id').alias('m_id'), F.lit(True).alias('member_exists')),
          F.col('member_id') == F.col('m_id'), 'left')

validated_df = validated_df.withColumn(
    'reject_reason',
    F.when(
        F.col('charger_id').isNull() | F.col('charger_exists').isNull(),
        F.lit('UNKNOWN_CHARGER')
    ).when(
        F.col('member_id').isNotNull() & F.col('member_exists').isNull(),
        F.lit('UNKNOWN_MEMBER')
    ).when(
        F.col('started_at').isNull(),
        F.lit('INVALID_TIMESTAMP')
    ).when(
        F.col('status').isin('COMPLETED', 'FAILED') & (
            F.col('ended_at').isNull() | (F.col('ended_at') <= F.col('started_at'))
        ), 
        F.lit('INVALID_DURATION')
    ).when(
        (F.col('status') == 'COMPLETED') & (
            F.col('energy_kwh').isNull() | (F.col('energy_kwh') < 0)
        ),
        F.lit('INVALID_METER')
    ).otherwise(F.lit(None))
)

###C4

In [0]:
silver_sessions_df = validated_df.filter(F.col('reject_reason').isNull()).select(
    'session_id', 'charger_id', 'member_id', 'started_at', 'ended_at', 
    'session_date', 'status', 'time_band', 'energy_kwh', 'duration_minutes', 
    'idle_minutes', F.col('payment_amount_inr').alias('amount_paid'), 'updated_at'
)

valid_with_faults = validated_df.filter(F.col('reject_reason').isNull()).select(
    'session_id',
    F.posexplode_outer(F.col('faults')).alias('fault_pos', 'fault_code')
)

silver_session_faults_df = valid_with_faults \
    .filter(F.col('fault_pos').isNotNull()) \
    .select(
        F.col('session_id'),
        (F.col('fault_pos') + 1).cast(IntegerType()).alias('fault_seq'),
        F.upper(F.trim(F.col('fault_code'))).alias('fault_code')
    )

quarantine_sessions_df = validated_df.filter(F.col('reject_reason').isNotNull()).select(
    'session_id', 'reject_reason', 'updated_at', 'raw_record',
    F.current_timestamp().alias('quarantined_at')
)

In [0]:
silver_sessions_df.createOrReplaceTempView("incoming_sessions")
silver_session_faults_df.createOrReplaceTempView("incoming_faults")
quarantine_sessions_df.createOrReplaceTempView("incoming_quarantine")

In [0]:
# 1. If incoming sessions are valid, remove them from quarantine (handles Quarantine -> Silver promotion)
spark.sql(f"""
    DELETE FROM {catalog}.silver.quarantine_sessions
    WHERE session_id IN (SELECT session_id FROM incoming_sessions)
""")

# 2. Merge into silver.sessions
spark.sql(f"""
    MERGE INTO {catalog}.silver.sessions AS target
    USING incoming_sessions AS source
    ON target.session_id = source.session_id
    WHEN MATCHED AND source.updated_at >= target.updated_at THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
""")

# 3. Refresh faults for updated valid sessions
spark.sql(f"""
    DELETE FROM {catalog}.silver.session_faults
    WHERE session_id IN (SELECT session_id FROM incoming_sessions)
""")

spark.sql(f"""
    INSERT INTO {catalog}.silver.session_faults
    SELECT * FROM incoming_faults
""")

# 4. If incoming sessions are quarantined, remove them from silver (handles Silver -> Quarantine demotion)
spark.sql(f"""
    DELETE FROM {catalog}.silver.sessions
    WHERE session_id IN (SELECT session_id FROM incoming_quarantine)
""")
spark.sql(f"""
    DELETE FROM {catalog}.silver.session_faults
    WHERE session_id IN (SELECT session_id FROM incoming_quarantine)
""")

# 5. Merge into silver.quarantine_sessions
spark.sql(f"""
    MERGE INTO {catalog}.silver.quarantine_sessions AS target
    USING incoming_quarantine AS source
    ON target.session_id = source.session_id
    WHEN MATCHED AND source.updated_at >= target.updated_at THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
""")